# Report (7068566, Team#3)

## 1. Data analysis & preprocessing

In this step, you will prepare the dataset for machine learning modeling. Carefully inspect each feature and decide how it should be transformed, encoded, or removed. Your goal is to convert all relevant variables into a clean numerical format while preserving useful information and avoiding data leakage. For this section, *focus specifically on the regression task*. If any preprocessing decisions would differ for the classification task, clearly indicate and justify those differences in separate cells.

### 1.1 Exploratory Data Analysis

In this section, include the key data analyses and visualizations that support your understanding of the dataset and inform your preprocessing decisions. Focus on extracting insights such as feature distributions, relationships between variables, missing data patterns, and potential anomalies.

You may also include exploratory or unsupervised techniques (e.g., clustering, PCA, dimensionality reduction) if they help reveal structure in the data or support your decisions.

The objective is to provide **evidence for the choices made in the Data preprocessing subsection**. Only include analyses that are directly relevant.

***Do not include raw data dumps or trivial visualizations (e.g., full tables of the dataset or plots of every feature).***
Keep this section concise and focused in the final version, ***even if more extensive exploration was performed during development***. *This section should include information on both tasks*.


##### Observations

Summarize the key findings and insights obtained from your exploratory data analysis. Highlight notable patterns, relationships, distributions, missing values, outliers, or any characteristics of the data that may influence preprocessing decisions and model performance.

**Free Text Form**

- `Kaggle_ID` and `Record_id` are pure row identifiers with no relationship to any other feature; both were dropped in every notebook.
- `Seasons` contains distinct misspelled variants of the 4 real season names, all corrected with a regex-based mapping (`fix_season` as main consonants are same) down to `Spring`, `Summer`, `Autumn`, `Winter`.
- `Hour` contains invalid negative values in both tasks; these carry no usable information (no correlation to other features) and were filtered out (`Hour >= 0`).
- `Wind speed` also contains invalid negative values (observed range: -2.0 to 15.7 m/s); rows with `Wind speed < 0` were filtered out.
- Regression training data size after filtering has dropped from 127,880 to 116,472 (size) which is quite low
- Only 8 observations in the regression data have `Wind speed > 8` m/s; these were inspected as potential high-leverage points but not removed by the final `filter_invalid` pipeline.
- In the classification data (which is *not* row-dropped up front), missing values are present in `Temperature`, `Solar Radiation`, and `Rainfall`; these are imputed per-day via `WeatherImputer` (interpolate - forward-fill - back-fill, grouped by `Date`).
- Correlation analysis shows `Rented Bike Count` is most strongly associated with `Temperature`, `Humidity`, and `Hour`; `Temperature x Humidity` and `Temperature rolling 3h avg` also correlate strongly with `Temperature` itself (as expected, since they are derived from it).
- The classification target `Demand_Category` is imbalanced across its three classes: out-of-fold prediction totals (from the confusion matrix) show class distribution of roughly 1531 / 3078 / 1558 observations for classes 0 / 1 / 2 respectively — the middle class is about twice as frequent as each of the others.



### 1.2 Data preprocessing

For each feature, select the appropriate preprocessing actions based on its type (numerical, categorical, temporal, or identifier-based). You may scale numerical variables, encode categorical variables, extract useful components from dates, handle missing values, and remove irrelevant or redundant columns. When multiple options are possible, justify your choices briefly based on how they would impact model performance. **The justification should usually be based on your data analyses**. 

> *Select all steps that you followed. Note that we distinguish between outliers (extreme but valid values) and invalid/impossible values (data errors).*

---

<table style="width:100%; table-layout:fixed;">
<tr>

<td style="width:50%; vertical-align:top; padding-right:20px;">

### Kaggle_ID

* [ ] Keep as is
* [X] Drop feature
* [ ] Use only for indexing
* [ ] Scale (standardize/normalize)
* [ ] Handle missing values
* [ ] Handle invalid/impossible values
* [ ] Handle outliers
* [ ] Other: ______

**Justification (max 2–3 sentences):**
Id doesn't bring any pattern forming information

---

### Record_id

* [ ] Keep as is
* [X] Drop feature
* [ ] Use only for indexing
* [ ] Scale (standardize/normalize)
* [ ] Handle missing values
* [ ] Handle invalid/impossible values
* [ ] Handle outliers
* [ ] Other: ______

**Justification (max 2–3 sentences):**
Id doesn't bring any pattern forming information

---

### Date

* [ ] Keep as is
* [X] Drop feature
* [X] Convert format (datetime)
* [ ] Extract components (day, month, year)
* [ ] Encode cyclically
* [ ] Bin into categories
* [ ] Handle missing values
* [ ] Handle invalid/impossible values
* [ ] Handle outliers
* [ ] Other: ______


**Justification (max 2–3 sentences):**
Converted to datetime (combined with `Hour`) purely to establish chronological order for feature engineering (rolling averages, 3h temperature change, rainfall accumulation, sorting); the raw `Date` column itself is dropped afterwards (`drop_unnecessary_columns`) since it does not generalize to unseen dates.

---

### Hour

* [X] Keep as is
* [ ] Drop feature
* [X] Scale (standardize/normalize)
* [ ] Convert format
* [X] Encode cyclically
* [ ] Bin into categories
* [ ] Handle missing values
* [X] Handle invalid/impossible values
* [ ] Handle outliers
* [ ] Other: ______

**Justification (max 2–3 sentences):**
Values under 0 are invalid, and in combination with other features don't have any information regardless of magnitude of negative value: -1 is same as -13 or -18

---

### Temperature

* [ ] Keep as is
* [ ] Drop feature
* [X] Scale (standardize/normalize)
* [ ] Transform distribution
* [ ] Bin into categories
* [X] Handle missing values
* [ ] Handle invalid/impossible values
* [ ] Handle outliers
* [ ] Other: ______

**Justification (max 2–3 sentences):**
Standardized like the other numeric features before feeding linear/SVM models and rows with missing `Temperature` imputed (interpolated) per-day in the classification pipeline.

---

### Humidity

* [ ] Keep as is
* [ ] Drop feature
* [X] Scale (standardize/normalize)
* [ ] Transform distribution
* [ ] Bin into categories
* [ ] Handle missing values
* [ ] Handle invalid/impossible values
* [ ] Handle outliers
* [ ] Other: ______

**Justification (max 2–3 sentences):**
Standardized along with the other numeric features; no invalid values were found in its observed range (0–98).

---

### Wind Speed

* [ ] Keep as is
* [ ] Drop feature
* [X] Scale (standardize/normalize)
* [ ] Transform distribution
* [ ] Bin into categories
* [ ] Handle missing values
* [X] Handle invalid/impossible values
* [ ] Handle outliers
* [ ] Other: ______

**Justification (max 2–3 sentences):**
Negative values are invalid and were filtered out and the remaining values are standardized. Values above 8 m/s (8 observations) we flagged as potential high-leverage points but did not remove as it may bear information for genralization.

---

</td>

<td style="width:50%; vertical-align:top; padding-left:20px;">

### Visibility

* [ ] Keep as is
* [ ] Drop feature
* [X] Scale (standardize/normalize)
* [ ] Transform distribution
* [ ] Bin into categories
* [ ] Handle missing values
* [ ] Handle invalid/impossible values
* [ ] Handle outliers
* [ ] Other: ______

**Justification (max 2–3 sentences):**
Standardized along with the other numeric features and values that appeared to be invalid were kept as in its observed range (0–2500) are still realistic.

---

### Dew Point Temperature

* [ ] Keep as is
* [ ] Drop feature
* [X] Scale (standardize/normalize)
* [ ] Transform distribution
* [ ] Bin into categories
* [ ] Handle missing values
* [ ] Handle invalid/impossible values
* [ ] Handle outliers
* [ ] Other: ______

**Justification (max 2–3 sentences):**
Standardized and also used as the basis for the engineered `Dew Point Depression`, `Wind Chill`, and `Fog Risk` features.

---

### Solar Radiation

* [ ] Keep as is
* [ ] Drop feature
* [X] Scale (standardize/normalize)
* [ ] Transform distribution
* [ ] Bin into categories
* [X] Handle missing values
* [ ] Handle invalid/impossible values
* [ ] Handle outliers
* [ ] Other: ______

**Justification (max 2–3 sentences):**
Standardized and rows with missing `Solar Radiation` are interpolated per-day in the classification pipeline.

---

### Rainfall

* [ ] Keep as is
* [ ] Drop feature
* [X] Scale (standardize/normalize)
* [ ] Transform distribution
* [ ] Bin into categories
* [X] Handle missing values
* [ ] Handle invalid/impossible values
* [ ] Handle outliers
* [ ] Other: ______

**Justification (max 2–3 sentences):**
Standardized; rows with missing `Rainfall` are imputed per-day also used as the basis for the engineered `Rainfall accumulation 3h` feature. All interpolations seemed reasonable as nature is quite smooth.

---

### Snowfall

* [ ] Keep as is
* [ ] Drop feature
* [X] Scale (standardize/normalize)
* [ ] Transform distribution
* [ ] Bin into categories
* [ ] Handle missing values
* [ ] Handle invalid/impossible values
* [ ] Handle outliers
* [ ] Other: ______

**Justification (max 2–3 sentences):**
Standardized along with the other numeric features. No missing or invalid values were found for this column.

---

### Seasons

* [ ] Keep as is
* [ ] Drop feature
* [X] Encode (one-hot / ordinal)
* [ ] Handle missing values
* [X] Handle invalid/impossible values
* [ ] Other: ______

**Justification (max 2–3 sentences):**
Raw values contain 16 misspelled variants of the 4 real seasons; corrected via a regex mapping (`fix_season`) before one-hot encoding.

---

### Holiday

* [ ] Keep as is
* [ ] Drop feature
* [X] Encode (one-hot / ordinal)
* [ ] Handle missing values
* [ ] Handle invalid/impossible values
* [ ] Other: ______

**Justification (max 2–3 sentences):**
Binary categorical feature (`No Holiday` / `Holiday`), one-hot encoded but `drop="first"` keeps only one anyway.

---

### Functioning Day

* [ ] Keep as is
* [ ] Drop feature
* [X] Encode (one-hot / ordinal)
* [ ] Handle missing values
* [ ] Handle invalid/impossible values
* [ ] Other: ______

**Justification (max 2–3 sentences):**
Binary categorical feature, one-hot encoded. Same as holiday.

---


#### (Optional) Classification-specific preprocessing

Describe any preprocessing steps applied specifically to the classification task that differ from the regression pipeline. Include a short explanation and provide the corresponding code below.

**Free Text Form**
- Missing values in `Temperature`, `Solar Radiation`, and `Rainfall` are not dropped. We used `WeatherImputer`  that interpolates points in the same day. followed by forward-fill and back-fill. This preserves rows instead of discarding them, since the classification training set is smaller.
- No engineered features
---





### 1.3 Feature Engineering

In this step, you should describe any new features that were created from the original dataset. If no new features were created, explicitly state that the original features were used without modification beyond preprocessing. ***"New" features made via dimensionality reduction methods such as PCA should be indicated here***.

**Did you create new features?**

* [X] Yes
* [ ] No

---

If YES, list your engineered features below:

**Example format:**
* Feature 1: __________________________
* How it was created: ___________________________________________
* Used in which of the tasks: _____________________________________________
* Why it is useful: _____________________________________________

**Engineered features actually used:**

* Feature: `Dew Point Depression`
  * How it was created: `Temperature - Dew point temperature`
  * Used in which of the tasks: Regression only
  * Why it is useful: how far the air temperature is above the dew point; small values (close to 0) mean the air is close to saturation.

* Feature: `Temperature x Humidity`
  * How it was created: `Temperature * Humidity`
  * Used in which of the tasks: Regression only
  * Why it is useful: interaction term between temperature and humidity.

* Feature: `Wind Chill`
  * How it was created: Environment-Canada-style metric wind-chill formula;
  * Used in which of the tasks: Regression only
  * Why it is useful: combines temperature and wind speed into a single perceived-temperature feature.

* Feature: `Hour sin` / `Hour cos`
  * How it was created: `sin(2π·Hour/24)` and `cos(2π·Hour/24)` (cyclical encoding); the raw `Hour` column is kept as well.
  * Used in which of the tasks: Regression only
  * Why it is useful: encodes hour-of-day as a point on a circle so hour 23 and hour 0 are close together, unlike the raw integer. Kept as try-out since weather is dependant on exact hour of the day and 1 to 23 arent actually far.

* Feature: `Fog Risk`
  * How it was created: binary indicator, `1` if `(Temperature - Dew point temperature) <= 2.5` and `Humidity >= 90`, else `0`.
  * Used in which of the tasks: Regression only
  * Why it is useful: fog tends to form when air is in saturation.

* Feature: `Rainfall accumulation 3h`
  * How it was created: rolling sum of `Rainfall` over the 3 hours before it, in chronological (Date + Hour) order.
  * Used in which of the tasks: Regression only
  * Why it is useful: captures recent rainfall accumulation and allows bikers to fall :) .

* Feature: `Temperature change 3h`
  * How it was created: difference between the current `Temperature` and the `Temperature` 3 hours earlier.
  * Used in which of the tasks: Regression only
  * Why it is useful: captures short-term temperature trend.

* Feature: `Temperature rolling 3h avg` / `Wind speed rolling 3h avg`
  * How it was created: rolling mean of `Temperature` and `Wind speed` over 3 hours before it.
  * Used in which of the tasks: Regression only
  * Why it is useful: smooths short-term noise in temperature and wind speed.

Note: none of these engineered features are computed for the classification task — `classification_demand_category.ipynb` does not call `engineer_features` and uses only the original columns.



## 2. ML modeling

In this step, you will explore and evaluate different machine learning model families to identify the most suitable approach for the problem. You are expected to experiment with multiple models, including baseline methods and more advanced algorithms. Generally, model selection should be guided by performance, interpretability, suitability for the dataset characteristics and other desiderata.

You should first select the general families of models you plan to test. Then, document the specific models you actually implemented, including the library used and a brief justification of each choice (do not justify the library). It is important to include all attempts, even models that performed poorly, as this reflects the full exploration process.

---

### Model Families that you explored (check all that apply)

* [X] Linear Models (e.g., Linear Regression, Logistic Regression, Ridge, Lasso, Generalized Linear Models)
* [X] Tree-Based Models (e.g., Decision Trees, Random Forest, Extra Trees, Gradient Boosting, XGBoost, LightGBM)
* [X] Support Vector Machines (SVM / SVR)
* [ ] k-Nearest Neighbors (KNN)
* [ ] Neural Networks / Deep Learning
* [ ] Ensemble Methods (e.g., Voting, Stacking, Bagging). If selected, state which models the ensemble consists of.
* [X] Probabilistic Models (e.g., Naive Bayes)
* [ ] Other: __________

### List of specific models you explored


> *You will **not** be evaluated based on the number of models you explore, but primarily on the quality of your final model selection and the reasoning behind the models you explored and tested.*



**Example format for each specific model you explored:**

> **Model:** Random Forest
> **Library:** scikit-learn
> **Why I used it:** Good baseline for non-linear relationships and robust to feature scaling.
> **Result/Notes:** Performed well on validation set, reduced overfitting compared to decision tree.

**Regression — models actually run (`regression_rented_bike_count.ipynb`):**

> **Model:** Ridge Regression
> **Library:** scikit-learn
> **Why I used it:** it was already there, why not.
> **Result/Notes:** Best CV RMSE = 429.65, MSLE = 2.761.

> **Model:** XGBoost (`XGBRegressor`)
> **Library:** xgboost
> **Result/Notes:** Tuned via `GridSearchCV` over `n_estimators ∈ {100, 200}`, `learning_rate ∈ {0.05, 0.1}`, `max_depth ∈ {3, 5}` (5-fold CV, scoring `neg_root_mean_squared_error`). Best parameters: `n_estimators=200, learning_rate=0.1, max_depth=5`. Best CV RMSE = 239.34, MSLE = 0.826. Later it showed that it overfits on the data anyway, so we forced it to max_depth=3 at some point. Didn't seem good anyway

**Regression — models defined but not actually fit (commented out in the notebook):**

> **Model:** Gradient Boosting Regressor
> **Library:** scikit-learn
> **Result/Notes:** Was tried out but no meaningful result.

> **Model:** Support Vector Regression (`SVR`, RBF kernel)
> **Library:** scikit-learn
> **Result/Notes:** Was tried out but no meaningful result.

**Classification — models actually run (`classification_demand_category.ipynb`), compared via 5-fold `StratifiedGroupKFold` (grouped by `Date`), scoring `f1_macro`:**

> **Model:** Logistic Regression
> **Library:** scikit-learn
> **Result/Notes:** `max_iter=2000`, default regularization. Mean CV macro F1 = 0.7154 (std 0.0196). Was already there.

> **Model:** Gaussian Mixture Model classifier (`GMMClassifier`, one `GaussianMixture` per class)
> **Library:** scikit-learn (`GaussianMixture`) wrapped in a custom `BaseEstimator`/`ClassifierMixin`
> **Result/Notes:** `n_components=3`, `covariance_type="full"`. Mean CV macro F1 = 0.5567 (std 0.0549); fold scores [0.6349, 0.5673, 0.5846, 0.4731, 0.5238]. Left out as the weakest of the four models tested. Tried it because classes seem to follow Gaussian disrtibution with histograms on pairplots showing difference in mean and variance.

> **Model:** Quadratic Discriminant Analysis (QDA)
> **Library:** scikit-learn
> **Result/Notes:** `reg_param=0.1`, mean F1 ranged from ~0.57 to ~0.66 across the other values tested). Mean CV macro F1 = 0.6024 (std 0.0404); fold scores [0.5979, 0.6500, 0.5320, 0.5996, 0.6325]. Tried it because it was natural reduction from same hypothesis as GMM.

> **Model:** Histogram-based Gradient Boosting Classifier (`HistGradientBoostingClassifier`)
> **Library:** scikit-learn
> **Result/Notes:** Mean CV macro F1 = 0.8196 (std 0.0178); fold scores [0.8370, 0.8106, 0.8038, 0.8061, 0.8407] — the best of the four models tested. Gradient boosting was natural with hope for better results. Kept for good generalizability and perfomance.





#### Final Selection - Regression

* Best performing model: XGBoost (`XGBRegressor`, `n_estimators=200, learning_rate=0.1, max_depth=5`)
* Reason for final choice (1–3 sentences):
It had the lowest 5-fold CV RMSE (239.34 vs. 429.65 for Ridge) and lowest MSLE (0.826 vs. 2.761 for Ridge).



#### Final Selection - Classification

* Best performing model: Histogram-based Gradient Boosting Classifier (`HistGradientBoostingClassifier`)
* Reason for final choice (1–3 sentences):
Selected automatically as the model with the highest mean of the 5-fold `StratifiedGroupKFold` macro-F1 scores (0.8196), ahead of Logistic Regression (0.7154), QDA (0.6024), and the GMM classifier (0.5567). It naturally captures non-linear relationships and feature interactions.



## 3. Model Selection

Briefly explain how you compared different models and selected your final pipeline. Include what criteria you used (e.g., validation score, interpretability, overfitting, computational cost). The answers here will be split into two; one for regression and one for classification.



### 3.1 Model Selection Strategy

Specify which models (or model families) were ultimately selected for final hyperparameter tuning and why they were chosen as the most promising candidates.

**Text - Regression (max 5–8 sentences):**
Ridge Regression (linear baseline), Gradient Boosting and XGBoost (tree-based, gradient-boosted ensembles), and SVR with an RBF kernel (non-linear margin-based regressor), were carried through to GridSearchCV hyperparameter tuning each tuned via 5-fold cross-validation with negative root mean squared error scoring. Gradient Boosting and XGBoost were chosen because tree-based ensembles handle the non-linear relationships and feature interactions seen in the weather data (e.g. temperature, humidity, wind speed), and SVR was included to test whether a non-linear, margin-based approach could capture smoother relationships that trees might overfit or interpolate around.

---

**Text - Classification (max 5–8 sentences):**
Logistic Regression (linear baseline), a custom Gaussian-Mixture-based generative classifier (GMMClassifier), Quadratic Discriminant Analysis, and Histogram-based Gradient Boosting (HistGradientBoostingClassifier), each evaluated with 5-fold StratifiedGroupKFold cross-validation using macro-F1 scoring. Logistic Regression was included as a simple, interpretable linear baseline and basically establishes how separable the three demand classes are along linear decision boundaries. The GMM classifier and QDA were chosen to test generative, distribution-based approaches, modeling each class as a Gaussian (mixture) in feature space to see whether the classes are well-described by such distributions. Histogram-based Gradient Boosting was included because tree-based ensembles can capture non-linear class boundaries and non-linear interactions between weather features without requiring those distributional assumptions.

<hr style="border: 2px solid white;">

### 3.2 Hyperparameter Selection Strategy

Explain which hyperparameters you chose to tune per model (e.g., regularization strength, tree depth, learning rate). Justify why these parameters are important and how they influence overfitting/underfitting and model performance. Additionally, explain/mention which searching algorithm you used.

**Text - Regression (max 5–8 sentences):** 
Ridge: `alpha ∈ {0.01, 0.1, 1, 10, 100}` (regularization strength: to control overfitting). Gradient Boosting: `n_estimators ∈ {100, 200}`, `learning_rate ∈ {0.05, 0.1}`, `max_depth ∈ {2, 3}` (number of boosting stages, step size per stage, and tree depth, control how much the ensemble can fit. Having more estimators, higher learning rate, or deeper trees all increase overfitting risk). XGBoost: `n_estimators ∈ {100, 200}`, `learning_rate ∈ {0.05, 0.1}`, `max_depth ∈ {1, 3, 5}` (the same overfitting/underfitting trade-off as Gradient Boosting, with tree depth to catch deeper feature interactions). SVR: `C ∈ {1, 10, 100}`, `gamma ∈ {"scale", 0.01, 0.1}`, `epsilon ∈ {0.1, 1}` (`C` trades off margin width , `gamma` controls how far a single training point reaches, and `epsilon` sets the width of the no-penalty around predictions, so they controll how tightly the model fits data). All four were tuned with `GridSearchCV`, 5-fold cross-validation, scoring `neg_root_mean_squared_error`.
---


**Text - Classification (max 5–8 sentences):**
No hyperparameter grid was run: Logistic Regression, GMM classifier, and HistGradientBoostingClassifier were all evaluated with their default hyperparameters (apart from `max_iter=2000` for Logistic Regression and GMM classifier 3 components). QDA's `reg_param` was tuned informally by manually testing values in `{0.0, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75}`, settling on `reg_param=0.1` for the final pipeline. All models were compared using 5-fold `StratifiedGroupKFold` (grouped by `Date` as it fails grouped interpolation otherwise), scoring `f1_macro`.

<hr style="border: 2px solid white;">


### 3.3 Model Evaluation

##### Evaluation Metrics (check all that apply)

* [ ] Accuracy
* [ ] Precision
* [ ] Recall
* [X] (macro) F1-score
* [ ] (micro) F1-score
* [ ] ROC-AUC
* [ ] Mean Absolute Error (MAE)
* [X] MSE / RMSE
* [ ] R² Score
* [ ] Mean Absolute Percentage Error (MAPE)
* [X] Mean Squared Logarithmic Error (MSLE)
* [ ] Log Loss
* [ ] Other: __________

Regression: RMSE (`neg_root_mean_squared_error`, used as the `GridSearchCV` scoring metric) and MSLE (computed on out-of-fold predictions via `cross_val_predict`, clipped to non-negative before the log). Classification: macro F1-score (`f1_macro`, used for CV  scoring metric); a confusion matrix was also inspected for the final classification model.

<hr style="border: 2px solid white;">

### (Optional) Additional Evaluation Criteria

Explain any additional criteria used to compare models beyond metrics (e.g., interpretability, robustness, speed, fairness, stability).

**Text - Regression (max 2–3 sentences):**


---


**Text - Classification (max 2–3 sentences):**
A confusion matrix on out-of-fold predictions of the final pipeline to inspect per-class misclassification patterns.

<hr style="border: 2px solid white;">

### 3.4 Final Pipeline

Provide a clear description of your final pipelines. You may also visualize them. Clearly indicate which pipeline corresponds to each task.

**Regression pipeline:** `ColumnTransformer` with (1) a numeric branch (`StandardScaler`) applied to all numeric/engineered columns, (2) a categorical branch (`fix_season` then `OneHotEncoder(drop="first")`) applied to `Seasons`, `Holiday`, `Functioning Day` — followed by `XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=5)`.

**Classification pipeline:** `WeatherImputer` (per-day interpolation of `Temperature`, `Solar Radiation`, `Rainfall`) into `ColumnTransformer` with (1) a numeric branch (`StandardScaler`) and (2) a categorical branch (`fix_season` then `OneHotEncoder(drop="first")`) applied to `Seasons`, `Holiday`, `Functioning Day` — followed by `HistGradientBoostingClassifier`.


#### Recommended format: sklearn-pipeline visualization (Jupyter)
If "model" is the variable storing your scikit-learn pipeline:

```python
from sklearn import set_config
set_config(display="diagram")

model
```

---



### 4. Empirical Results

In this section, you summarize the key experimental results obtained during model development. Focus on how different models and configurations performed on training and validation data, and how these results informed the selection of the final model. The goal is to clearly show evidence-based decision making rather than subjective choice.

---

#### 4.1 Summary of Main Results

Provide a concise overview of the most relevant experimental outcomes that led to selecting the final model. Highlight performance differences between models and any notable trends (e.g., overfitting, stability, or consistent improvements from tuning).

**Text - Regression (max 5–8 sentences):**
XGBoost showed a clear and consistent improvement over Ridge Regression across both CV RMSE (239.34 vs. 429.65) and MSLE (0.826 vs. 2.761), reflecting the trend that tree-based ensembles models capture non-linear feature interactions well, better than a linear model. Gradient Boosting was expected to perform similarly to XGBoost, since both build sequential ensembles of shallow trees to correct prior errors, so it was assumed to improve stability from an independent implementation and a lower risk of overfitting than XGBoost. SVR with an RBF kernel, was suited to capture smooth continuous non-linear relationships without the sometimes-weird step boundaries typical of tree splits, at the cost of higher sensitivity to feature scaling and slower training with more training points. In practice, however, the improvements-from-tuning trend mattered most and the gap between the linear and non-linear model families, and XGBoost's had the strongest performance

---


**Text - Classification (max 5–8 sentences):**
The generative approaches performed weakest — the GMM classifier scored a mean macro F1 of 0.5567 and QDA scored 0.6024 — suggesting the distributions are not super well approximated by Gaussians. Logistic Regression, as the linear baseline, scored notably higher at 0.7154, indicating the classes are partly linearly separable. HistGradientBoostingClassifier clearly outperformed all other candidates with a mean macro F1 of 0.8196, reflecting its ability to capture non-linear interactions between weather features without requiring distributional assumptions. Based on this consistent and substantial margin over every other candidate, HistGradientBoostingClassifier was selected as the final classification model.


<hr style="border: 2px solid white;">

#### 4.2 Final Model Justification

Justify why the final model was selected based on empirical evidence. Keep the explanation focused on performance and practical considerations.

**Text - Regression (max 5–8 sentences):**
XGBoost had both the lowest CV RMSE (239.34 vs. 429.65 for Ridge) and the lowest CV MSLE (0.826 vs. 2.761 for Ridge)

---


**Text - Classification (max 5–8 sentences):**
`HistGradientBoostingClassifier` had the highest mean CV macro F1 (0.8196) among all four models tested, and was selected as the top row of the mean-F1-sorted comparison table.


<hr style="border: 2px solid white;">

#### 4.3 Take-home Messages

Summarize the key insights learned from your experiments. This may include what worked well, what did not, and any unexpected findings.

**Text - Regression (max 5–8 sentences):**
Tree-based models (XGBoost for regression, HistGradientBoostingClassifier for classification) outperformed the corresponding linear/probabilistic baselines (Ridge; Logistic Regression, QDA, GMM) by a wide difference on both tasks. For classification, the GMM scored lowest of all four models tested, and QDA as well even though classes seemed to be gaussian.

---


**Text - Classification (max 5–8 sentences):**
The classification confusion matrix shows the model separates the smaller classes (0 and 2) fairly well from each other but confuses each of them somewhat with the larger middle class.


<hr style="border: 2px solid white;">

#### 4.4 Final Performance Summary

Report the final performance metrics of your selected model. If you created your own test split from the original training data (i.e., train/validation/test split), include the corresponding results below. If no separate test set was used, you may leave that row blank or omit it.

<table>
<tr>
<td><strong>Regression</strong></td>
<td><strong>Classification</strong></td>
</tr>

<tr>
<td>

| Dataset / Source              | Metric                   | Score |
| ----------------------------- | ------------------------ | ----- |
| Training set                  | —                        | in notebook |
| Validation set                | RMSE / MSLE (5-fold CV)  | RMSE = 239.34 / MSLE = 0.826 |
| Local Test set (if available) | —                        | no |
| Kaggle Leaderboard            | —                        | 0.60 |

</td>

<td>

| Dataset / Source              | Metric                   | Score |
| ----------------------------- | ------------------------ | ----- |
| Training set                  | —                        | in notebook |
| Validation set                | macro F1 (5-fold CV)     | 0.8196 |
| Local Test set (if available) | —                        | no |
| Kaggle Leaderboard            | —                        | 0.60 |

</td>
</tr>
</table>


---

If applicable, briefly comment on generalization performance (e.g., train vs validation gap, leaderboard consistency).

**Optional notes:**
No separate held-out test split was created for either task; both model comparisons rely entirely on 5-fold cross-validation (plain `KFold`/`GridSearchCV` for regression, `StratifiedGroupKFold` grouped by `Date` for classification), so the scores above are cross-validated estimates rather than a held-out test score.
---

##### Sample Submission Code
```python
test = pd.read_csv("test.csv")
test_X = custom_preprocess(test_X, None, is_test=True)
preds = final_pipeline.predict(test_X)
submission = pd.read_csv("sample_submission.csv")
target_variable = "Rented Bike Count"

# Fill target column
submission[target_variable] = preds

# Save submission file
submission.to_csv("my_submission.csv", index=False)

print("submission created successfully")
print(submission.head())
```


### 5. Additional Analysis (Optional)

In this section, you may include any further analysis that goes beyond the core modeling pipeline. This is an opportunity to demonstrate deeper understanding of the data, the model behavior, or the underlying structure of the problem. Any additional insights, visualizations, or experiments are welcome as long as they are clearly explained. Here we provide some examples:

---

1) *Analysis of Predictions*: Analyze the models' predictions in more detail. This may include error analysis, inspection of failure cases, residual plots, or comparison between predicted and actual values. Highlight any patterns you observe.
2) *Interpretability/Explainability*: Include an interpretation of your deployed models. This may involve feature importance, coefficients, SHAP values, or any other post-hoc explainability method used to understand model decisions. Fitting a separate, fully interpretable model also applies here.


#### Regression

- An exploratory `XGBRegressor` (`n_estimators=200, learning_rate=0.01, max_depth=5`, fit on the scaled/one-hot-encoded features) was overlaid as a binned fit line on pairplot panels involving `Rented Bike Count`, giving a visual sense of how the model's predictions track each feature; this exploratory model reached a training RMSE of 272.63 (note: this is a differently-configured, exploratory fit, not the final tuned `XGBRegressor` used for submission).
- Correlations were separately computed for the subset of 8 observations with `Wind speed > 8` m/s to check whether this small group behaves differently from the rest of the data.
- Rows where `Dew point temperature > Temperature` (8 rows, physically inconsistent) were isolated and inspected via the `Dew Point Depression` feature, which takes negative values for these rows.
- The final chosen model's predictions were plotted against each numeric feature (actual scatter vs. XGBoost fit line) to visually inspect the fitted relationships.



#### Classification

- Out-of-fold predictions from the Logistic Regression and GMM classifier pipelines (via `cross_val_predict` with the same `StratifiedGroupKFold` splits) were compared against the ground-truth `Demand_Category` labels using pairplots over the first 5 numeric features, to visually compare each model's decision behaviour against the true class structure.
- A confusion matrix was computed for the final `HistGradientBoostingClassifier` pipeline on out-of-fold predictions: `[[1207, 318, 6], [241, 2577, 260], [3, 275, 1280]]` (rows = true class, columns = predicted class), showing the model separates the two smaller classes from each other well but confuses each of them somewhat with the larger middle class.

